In [1]:
%connect -h localhost -p 6379
%status

server:   localhost:6379
db:       0
protocol: RESP3
tls:      off
user:     (default)
password: (none)
version:  8.6.2
mode:     standalone
server:   localhost:6379
db:       0
protocol: RESP3
tls:      off
user:     (default)
password: (none)
version:  8.6.2
mode:     standalone


In [2]:
"FT.DROP" "beers"
"FT.CREATE" "beers" "ON" "JSON" "PREFIX" "1" "brewery:" "SCORE" "1.0" "SCHEMA" "$.city" "AS" "city" "TAG" "SEPARATOR" "," "$.state" "AS" "state" "TAG" "SEPARATOR" "," "$.beers[*].style" "AS" "style" "TEXT" "WEIGHT" "1.0" "$.beers[*].abv" "AS" "abv" "NUMERIC" "$.beers[*].ibu" "AS" "ibu" "NUMERIC" "$.beers[*].ounces" "AS" "ounces" "NUMERIC"
# Use magic to load the data we're going to search
%load data/inputs.txt --quiet

OK
OK


In [3]:
%render rich
FT.AGGREGATE beers '*' GROUPBY 1 @state REDUCE COUNT 0 as Count SORTBY 2 @Count DESC MAX 5

state,Count
CO,47
CA,39
MI,32
OR,29
TX,28


In [4]:
%render plain 
FT.AGGREGATE beers '*' GROUPBY 1 @state REDUCE COUNT 0 as Count SORTBY 2 @Count DESC MAX 5

1# attributes => (empty array)
2# format => STRING
3# results => 
   1) 1# extra_attributes => 
         1# "state" => "CO"
         2# "Count" => "47"
      2# values => (empty array)
   2) 1# extra_attributes => 
         1# "state" => "CA"
         2# "Count" => "39"
      2# values => (empty array)
   3) 1# extra_attributes => 
         1# "state" => "MI"
         2# "Count" => "32"
      2# values => (empty array)
   4) 1# extra_attributes => 
         1# "state" => "OR"
         2# "Count" => "29"
      2# values => (empty array)
   5) 1# extra_attributes => 
         1# "state" => "TX"
         2# "Count" => "28"
      2# values => (empty array)
4# total_results => (integer) 51
5# warning => (empty array)


In [5]:
FT.SEARCH beers '@state:{RI} @abv:[(0.08 +inf]' RETURN 3 '$.beers[?(@.abv>0.08)].style' as Beer_Style DIALECT 3

id,Beer_Style
brewery:86,"[""American Double / Imperial IPA""]"
brewery:143,"[""German Pilsener""]"


In [6]:
FT.AGGREGATE beers '@state:{ME}' GROUPBY 1 @city REDUCE COUNT 0 as Breweries SORTBY 2 @Breweries DESC MAX 3

city,Breweries
Portland,6
Lewiston,1
Pineland,1


In [7]:
JSON.GET "brewery:177"

"{\"brewery_id\":\"177\",\"brewery_name\":\"18th Street Brewery\",\"city\":\"Gary\",\"state\":\"IN\",\"beers\":[{\"id\":\"2099\",\"abv\":0.072,\"ibu\":0,\"name\":\"Sophomoric Saison\",\"style\":\"Saison / Farmhouse Ale\",\"ounces\":12.0},{\"id\":\"2098\",\"abv\":0.073,\"ibu\":0,\"name\":\"Regional Ring Of Fire\",\"style\":\"Saison / Farmhouse Ale\",\"ounces\":12.0},{\"id\":\"2097\",\"abv\":0.069,\"ibu\":0,\"name\":\"Garce Sel\xc3\xa9\",\"style\":\"Saison / Farmhouse Ale\",\"ounces\":12.0},{\"id\":\"1980\",\"abv\":0.085,\"ibu\":0,\"name\":\"Troll Destroyer\",\"style\":\"Belgian IPA\",\"ounces\":12.0},{\"id\":\"1979\",\"abv\":0.061,\"ibu\":60.0,\"name\":\"Bitter Bitch\",\"style\":\"American Pale Ale (APA)\",\"ounces\":12.0}]}"
